# CodeRAG-Bench evaluation: open-domain (ODEX)

**ODEX** is CodeRAG-Bench's open-domain task. The query is a natural-language intent ("decode a hex string
`'4a4b4c'` to UTF-8"), and the corpus is 34,003 Python library
documentation snippets ([`neulab/docprompting-conala`](https://huggingface.co/datasets/neulab/docprompting-conala), the `docs`
config) -- nothing in the query is copied from the correct document. From
CodeRAG-Bench's own Table 3 (NDCG@10):

| Retriever | NDCG@10 |
|---|---|
| BM25 | 6.7 |
| GIST-base | 12.1 |
| Jina-v2-code | 19.9 |
| BGE-base | 22.0 |
| Voyage-code | 26.6 |
| GIST-large | 28.0 |
| SFR-Mistral | 37.1 |

A five-fold spread between the weakest and strongest retriever, instead of
everything clustering at the ceiling. This is the first task where QAChat's
retrieval quality is actually being tested.

This notebook reuses the scoring/chunking helpers from the previous notebook -- pulled out
into `benchmarks/common.py` so they aren't re-pasted into every benchmark
notebook -- rather than redefining them here.


**Important:** Since the original evaluation keeps every document whole — one document is one retrievable unit, never split — but QAChat's real pipeline chunks documents before indexing them, a single corpus document can become several chunks with new, content-based identifiers. The original `qrels` are keyed by whole-document id, so as soon as a gold document splits, its id no longer exists in the index and `evaluate_retrieval` would score that query as a miss even when retrieval found the right document. We fix this by propagating each document's relevance to every chunk produced from it, so a hit on *any* chunk of the gold document still counts.

## 1. Prepare QAChat

In [1]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


QAChat root: /Users/cmanzo/Documents/GitHub/QAChat


In [2]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [3]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


QAChat environment ready: /Users/cmanzo/Documents/GitHub/QAChat/.venv/bin/python


## 2. Benchmark-only dependencies

In [4]:
try:
    import datasets
    import numpy
    import pytrec_eval
except ImportError as error:
    raise RuntimeError(
        "Benchmark dependencies are missing. From the QAChat root, run:\n"
        "    uv sync --extra mlx --extra torch --extra bench\n"
        "(always pass every extra together) then select this notebook's\n"
        "kernel again."
    ) from error

print(f"datasets {datasets.__version__}, numpy {numpy.__version__}")


/Users/cmanzo/Documents/GitHub/QAChat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets 5.0.1, numpy 2.5.2


## 3. Build the ODEX task

Three sources, matching CodeRAG-Bench's own
[`create/odex.py`](https://github.com/code-rag-bench/code-rag-bench/blob/main/retrieval/create/odex.py):

- **Queries**: [`neulab/odex`](https://huggingface.co/datasets/neulab/odex)
  (`en` config, `test` split) -- each row's `intent` field.
- **Corpus**: `neulab/docprompting-conala` (`docs` config, `train` split)
  -- 34,003 `{doc_id, doc_content}` pairs.
- **Qrels**: a small canonical annotation file CodeRAG-Bench ships in its
  own repository, mapping each ODEX example to the corpus document(s) that
  answer it. It isn't distributed through `datasets.load_dataset`, so this
  fetches it directly (not vendored into this repo, since it's someone
  else's file with no stated license -- fetched at run time and used only
  as an evaluation annotation, the same way the rest of this notebook
  reaches out to Hugging Face).

Not every ODEX intent has an annotated relevant document -- only 201 of
439 do. The other 238 are dropped, matching CodeRAG-Bench's own practice
(and how `pytrec_eval` behaves anyway: a query absent from `qrels` is
silently excluded from scoring, not counted as a miss).


In [5]:
import json
import urllib.request

ODEX_DOCS_URL = (
    "https://raw.githubusercontent.com/code-rag-bench/code-rag-bench/"
    "main/retrieval/datasets/canonical/odex_docs.json"
)

with urllib.request.urlopen(ODEX_DOCS_URL) as response:
    odex_docs_text = response.read().decode("utf-8")

odex_docs_rows = [
    json.loads(line) for line in odex_docs_text.splitlines() if line.strip()
]
print(f"ODEX canonical rows: {len(odex_docs_rows)}")


ODEX canonical rows: 439


In [6]:
# neulab/docprompting-conala ships a legacy loading script, which
# modern `datasets` refuses to execute ("Dataset scripts are no
# longer supported"). Hugging Face auto-converts every dataset to
# Parquet regardless, so this loads that conversion directly instead
# of going through the repository's own script.
DOCPROMPTING_DOCS_PARQUET_URL = (
    "https://huggingface.co/datasets/neulab/docprompting-conala/"
    "resolve/refs%2Fconvert%2Fparquet/docs/train/0000.parquet"
)

doc_rows = datasets.load_dataset(
    "parquet", data_files=DOCPROMPTING_DOCS_PARQUET_URL
)["train"]

doc_index_by_id = {
    doc_id: index for index, doc_id in enumerate(doc_rows["doc_id"])
}

odex_corpus: dict[str, dict[str, str]] = {
    str(index): {"title": doc_id, "text": doc_content}
    for index, (doc_id, doc_content) in enumerate(
        zip(doc_rows["doc_id"], doc_rows["doc_content"])
    )
}
print(f"Documentation corpus: {len(odex_corpus)} documents")


Documentation corpus: 34003 documents


In [7]:
# Same Parquet-conversion workaround as the corpus load above --
# neulab/odex also ships a loading script.
ODEX_EN_TEST_PARQUET_URL = (
    "https://huggingface.co/datasets/neulab/odex/"
    "resolve/refs%2Fconvert%2Fparquet/en/test/0000.parquet"
)

odex_rows = datasets.load_dataset(
    "parquet", data_files=ODEX_EN_TEST_PARQUET_URL
)["train"]
assert len(odex_rows) == len(odex_docs_rows), (
    "neulab/odex[en][test] and the canonical annotation file must be in "
    "the same order -- row counts differ, something upstream changed."
)

odex_queries: dict[str, str] = {}
odex_qrels: dict[str, dict[str, int]] = {}
missing_titles = 0

for index, (row, annotated) in enumerate(zip(odex_rows, odex_docs_rows)):
    matched = {
        str(doc_index_by_id[doc["title"]]): 1
        for doc in annotated["docs"]
        if doc["title"] in doc_index_by_id
    }
    missing_titles += len(annotated["docs"]) - len(matched)
    if not matched:
        # No annotated doc, or none of its titles matched the corpus --
        # either way this query cannot be scored, so it is dropped rather
        # than kept with an empty qrels entry.
        continue

    query_id = f"{index}_{row['task_id']}"
    odex_queries[query_id] = row["intent"]
    odex_qrels[query_id] = matched

print(
    f"ODEX queries with a known relevant document: {len(odex_queries)} "
    f"of {len(odex_rows)}"
)
if missing_titles:
    print(
        f"Annotated doc titles not found in the corpus: {missing_titles} "
        "(dropped from qrels; check the corpus load above if this looks "
        "large)"
    )


ODEX queries with a known relevant document: 201 of 439


In [8]:
from benchmarks.common import BenchmarkTask

odex_task = BenchmarkTask(
    "odex", odex_corpus, odex_queries, odex_qrels
)
print(
    f"{odex_task.name}: {len(odex_task.queries)} queries, "
    f"{len(odex_task.corpus)} corpus documents"
)


odex: 201 queries, 34003 corpus documents


## 4. Load the embedding model

Same embedding-loading pattern as the previous notebook, and the same
instruction-formatted query -- `format_query`, imported from
`benchmarks.common` instead of redefined here.


In [9]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}


In [10]:
from qachat import choose_backend

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Model: {MODEL_ID}")


Backend: mlx
Model: mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ


In [11]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")


Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 2377.10it/s]


Loaded on: metal


## 5. Turn the corpus into a real QAChat project

`RAGSession._prepare_source` scans a project folder on disk. To exercise
that same path instead of a synthetic one-document-one-chunk mapping,
every corpus document is written out as its own `.txt` file under
`sandbox/odex_benchmark/repository/`, named after its corpus id so a
chunk's file path can be traced straight back to the document it came
from. `.txt` has no dedicated QAChat extension module, so it falls back
to the generic whole-file extractor -- the same real chunking pipeline
still applies, it just isn't Python- or Markdown-aware for this corpus.
`sandbox/odex_benchmark/profile.ini` is then opened as a real
`QAProject`.

A pathological-outlier problem applies before
writing to disk: a handful of ODEX documents run up to ~386,000
characters against a median of ~318. Real chunking turns
one such outlier into hundreds of chunks on its own, multiplying the
total embedding workload for content that compresses to noise anyway --
so `MAX_DOC_CHARS` can be used to cap document text before it becomes a file. Normal
documents are unaffected and now get real, multi-chunk coverage instead
of being compressed into one vector.


In [ ]:
# None disables truncation entirely. Expect a meaningfully larger odex_chunks 
# total and slower indexing without it.
MAX_DOC_CHARS: int | None = None

project_root = root / "sandbox" / "odex_benchmark"
profile_path = project_root / "profile.ini"
if not profile_path.exists():
    raise RuntimeError(
        f"Missing project profile: {profile_path}. This notebook expects "
        "the profile checked into the repository at that path."
    )

repository_dir = project_root / "repository"
repository_dir.mkdir(parents=True, exist_ok=True)

path_to_doc_id: dict[str, str] = {}
truncated = 0
for doc_id, doc in odex_task.corpus.items():
    text = doc["text"]
    if MAX_DOC_CHARS is not None and len(text) > MAX_DOC_CHARS:
        text = text[:MAX_DOC_CHARS]
        truncated += 1
    file_path = repository_dir / f"{doc_id}.txt"
    file_path.write_text(text)
    path_to_doc_id[file_path.name] = doc_id

cap_desc = "no cap" if MAX_DOC_CHARS is None else f"{MAX_DOC_CHARS} characters"
print(
    f"Wrote {len(path_to_doc_id)} corpus documents under {repository_dir} "
    f"({truncated} truncated to {cap_desc})."
)


Wrote 34003 corpus documents under /Users/cmanzo/Documents/GitHub/QAChat/sandbox/odex_benchmark/repository (0 truncated to no cap).


In [13]:
from qachat.project import open_project

project = open_project(project_root)
profile = project.profile

print(f"Project source: {project.source}")
print(
    f"Chunking: target={profile.target_tokens} max={profile.max_tokens} "
    f"overlap={profile.overlap_tokens}"
)


Project source: /Users/cmanzo/Documents/GitHub/QAChat/sandbox/odex_benchmark/repository
Chunking: target=300 max=400 overlap=40


In [14]:
from qachat.chunking import Chunk, chunk_source_units
from qachat.repository import scan_repository
from qachat.sources import extract_source_units, read_repository_files

# scan_repository defaults to max_entries=10_000 as a safety guard against
# accidentally pointing it at a huge or wrong folder. ODEX's repository has
# one file per corpus document, so we raise the limit deliberately to match
# the corpus we just wrote, rather than picking an arbitrary large number.
repository_report = scan_repository(
    project.source,
    extensions=profile.extensions,
    max_file_bytes=profile.max_file_bytes,
    max_entries=len(path_to_doc_id) + 100,
)
source_files = read_repository_files(repository_report)
source_units = extract_source_units(source_files)
odex_chunks = chunk_source_units(
    source_units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=profile.target_tokens,
    max_tokens=profile.max_tokens,
    overlap_tokens=profile.overlap_tokens,
)

print(
    f"{len(source_files)} files -> {len(source_units)} source units -> "
    f"{len(odex_chunks)} chunks"
)


34003 files -> 34003 source units -> 80520 chunks


## 6. Propagate document-level qrels to chunk level

Since the original ODEX qrels are keyed by whole-document id but real
chunking can split one document into several chunks with new,
content-based identifiers, a hit on any chunk of a gold document should
still count. Check first how many documents actually split.


In [15]:
chunks_by_doc_id: dict[str, list[Chunk]] = {}
chunk_by_id: dict[str, Chunk] = {}
for chunk in odex_chunks:
    chunk_by_id[chunk.identifier] = chunk
    doc_id = path_to_doc_id.get(chunk.path.name)
    if doc_id is None:
        continue
    chunks_by_doc_id.setdefault(doc_id, []).append(chunk)

doc_ids = list(odex_task.corpus)
counts = [len(chunks_by_doc_id.get(doc_id, [])) for doc_id in doc_ids]
split = sum(1 for count in counts if count > 1)
empty = sum(1 for count in counts if count == 0)
print(
    f"odex: {len(doc_ids)} documents, {split} split into >1 chunk, "
    f"{empty} produced 0 chunks (max chunks for one document: "
    f"{max(counts) if counts else 0})"
)


def propagate_qrels(
    qrels: dict[str, dict[str, int]],
) -> dict[str, dict[str, int]]:
    """Give every chunk of a relevant document that document's relevance."""

    qrels_chunks: dict[str, dict[str, int]] = {}
    for query_id, doc_scores in qrels.items():
        chunk_scores: dict[str, int] = {}
        for doc_id, relevance in doc_scores.items():
            for chunk in chunks_by_doc_id.get(doc_id, ()):
                chunk_scores[chunk.identifier] = relevance
        qrels_chunks[query_id] = chunk_scores
    return qrels_chunks


qrels_chunks: dict[str, dict[str, int]] = propagate_qrels(odex_task.qrels)

missing = sum(1 for scores in qrels_chunks.values() if not scores)
print(f"Queries left with zero relevant chunks after propagation: {missing}")


odex: 34003 documents, 5624 split into >1 chunk, 0 produced 0 chunks (max chunks for one document: 416)
Queries left with zero relevant chunks after propagation: 0


### Verify: every chunk of a gold document is actually covered

Not a scoring change -- a check that the propagation above did what it
claims. For every query, every chunk belonging to that query's gold
document should now carry the document's relevance in `qrels_chunks`,
whatever rank that chunk comes back at.


In [16]:
mismatches = []

# Coverage check, not set equality: a query can have several gold documents
# that share the same relevance value (ODEX qrels are effectively binary),
# so the chunks of one gold document can legitimately sit alongside chunks
# from another gold document at the same score in qrels_chunks[query_id].
# Comparing full sets against that shared score bucket produced false
# positives -- what actually matters is that every chunk of *this* document
# made it into qrels_chunks with the right relevance.
for query_id, doc_scores in odex_task.qrels.items():
    for doc_id, relevance in doc_scores.items():
        expected_chunk_ids = {
            chunk.identifier for chunk in chunks_by_doc_id.get(doc_id, ())
        }
        actual_scores = qrels_chunks[query_id]
        uncovered = {
            chunk_id
            for chunk_id in expected_chunk_ids
            if actual_scores.get(chunk_id) != relevance
        }
        if uncovered:
            mismatches.append((query_id, doc_id, uncovered))

if mismatches:
    raise AssertionError(
        f"{len(mismatches)} document(s) have chunks missing from "
        f"qrels_chunks after propagation -- first: {mismatches[0]}"
    )

print(
    "Every chunk of every gold document is present in qrels_chunks "
    "with the document's relevance."
)


Every chunk of every gold document is present in qrels_chunks with the document's relevance.


## 7. Run retrieval

`odex_chunks` is already the real, chunked corpus from section 4 --
nothing left to build here except the semantic and lexical indexes over
it, and the per-query rankings. `build_semantic_index(..., batch_size=)`
and `semantic_search(..., backend="numpy", matrix=...)` are the same
`qachat` capabilities used throughout.


In [17]:
EMBED_BATCH_SIZE = 32

from tqdm.auto import tqdm

from qachat import (
    build_lexical_index,
    build_semantic_index,
    lexical_search,
    semantic_matrix,
    semantic_search,
)

index_progress = tqdm(total=len(odex_chunks), desc="Embedding ODEX chunks")


def _report_progress(current: int, total: int) -> None:
    index_progress.n = current
    index_progress.refresh()


odex_semantic_index = build_semantic_index(
    odex_chunks,
    embedding_session,
    batch_size=EMBED_BATCH_SIZE,
    progress_callback=_report_progress,
)
index_progress.close()
odex_matrix = semantic_matrix(odex_semantic_index)
odex_lexical_index = build_lexical_index(odex_chunks)

print(f"Indexed {len(odex_chunks)} chunks for both channels.")


Embedding ODEX chunks: 100%|██████████| 80520/80520 [3:39:21<00:00,  6.12it/s]  


Indexed 80520 chunks for both channels.


In [ ]:
from tqdm.auto import tqdm

from benchmarks.common import format_query

SEARCH_COUNT = 100

semantic_ranked_by_query: dict[str, list] = {}
lexical_ranked_by_query: dict[str, list] = {}
semantic_results: dict[str, dict[str, float]] = {}
lexical_results: dict[str, dict[str, float]] = {}

for query_id, query_text in tqdm(
    odex_task.queries.items(), desc="Scoring ODEX queries"
):
    semantic_ranked = semantic_search(
        odex_semantic_index,
        format_query(query_text),
        embedding_session,
        count=SEARCH_COUNT,
        backend="numpy",
        matrix=odex_matrix,
    )
    lexical_ranked = lexical_search(
        odex_lexical_index, query_text, count=SEARCH_COUNT
    )
    semantic_ranked_by_query[query_id] = semantic_ranked
    lexical_ranked_by_query[query_id] = lexical_ranked
    semantic_results[query_id] = {
        result.chunk.identifier: result.score for result in semantic_ranked
    }
    lexical_results[query_id] = {
        result.chunk.identifier: result.score for result in lexical_ranked
    }

print(f"Scored {len(semantic_results)} queries.")


Scoring ODEX queries: 100%|██████████| 201/201 [00:57<00:00,  3.52it/s]

Scored 201 queries.


## 8. Fuse semantic and lexical for a hybrid run

`hybrid_search` takes each channel's ranked results directly -- the
real `SemanticResult`/`LexicalResult` objects `semantic_search` and
`lexical_search`, kept per query above, reused here rather than rebuilt 
from a plain score dict.


In [19]:
from qachat import hybrid_search

hybrid_results: dict[str, dict[str, float]] = {}
for query_id in odex_task.queries:
    fused = hybrid_search(
        semantic=semantic_ranked_by_query[query_id],
        lexical=lexical_ranked_by_query[query_id],
        count=SEARCH_COUNT,
    )
    hybrid_results[query_id] = {
        result.chunk.identifier: result.score for result in fused
    }

print(f"Fused {len(hybrid_results)} queries.")


Fused 201 queries.


## 9. Score against the published range

`evaluate_retrieval` imported from
`benchmarks.common`. Scored against `qrels_chunks` (section 6's
chunk-level propagation), not the original document-level `odex_task.qrels`: a hit on any chunk of a gold
document counts, since that document may now be split across several
chunks.


In [20]:
from benchmarks.common import evaluate_retrieval

odex_scores = {
    "semantic": evaluate_retrieval(qrels_chunks, semantic_results),
    "lexical": evaluate_retrieval(qrels_chunks, lexical_results),
    "hybrid": evaluate_retrieval(qrels_chunks, hybrid_results),
}

PUBLISHED_NDCG_AT_10 = {
    "BM25": 6.7,
    "GIST-base": 12.1,
    "Jina-v2-code": 19.9,
    "BGE-base": 22.0,
    "Voyage-code": 26.6,
    "GIST-large": 28.0,
    "SFR-Mistral": 37.1,
}

print("Published CodeRAG-Bench NDCG@10 (Table 3, document-level):")
for name, value in PUBLISHED_NDCG_AT_10.items():
    print(f"  {name:>14}: {value:5.1f}")

print("\nQAChat channels, this run (chunk-level, not directly comparable):")
for channel, scores in odex_scores.items():
    ndcg_10 = scores["ndcg"]["NDCG@10"] * 100
    mrr_10 = scores["mrr"]["MRR@10"]
    recall_10 = scores["recall"]["Recall@10"]
    print(
        f"  {channel:>14}: NDCG@10={ndcg_10:5.1f}  "
        f"MRR@10={mrr_10:.3f}  Recall@10={recall_10:.3f}"
    )


Published CodeRAG-Bench NDCG@10 (Table 3, document-level):
            BM25:   6.7
       GIST-base:  12.1
    Jina-v2-code:  19.9
        BGE-base:  22.0
     Voyage-code:  26.6
      GIST-large:  28.0
     SFR-Mistral:  37.1

QAChat channels, this run (chunk-level, not directly comparable):
        semantic: NDCG@10= 14.7  MRR@10=0.167  Recall@10=0.231
         lexical: NDCG@10=  3.8  MRR@10=0.048  Recall@10=0.059
          hybrid: NDCG@10= 10.6  MRR@10=0.126  Recall@10=0.152


### A worked example
The aggregate scores above show
*that* each channel is competitive; this picks one ODEX query where
lexical retrieval did not rank a gold chunk first, and shows what each
channel actually returned -- reusing `semantic_results`, `lexical_results`,
and `hybrid_results` from sections 7 and 8. Gold ids are now chunk ids, not document ids, so the displayed text comes from the chunk itself via `chunk_by_id`, not the original corpus entry.


In [21]:
DISPLAY_CHARS = 1500

example_query_id = None
for query_id, chunk_scores in qrels_chunks.items():
    gold_ids = {
        chunk_id for chunk_id, relevance in chunk_scores.items() if relevance > 0
    }
    doc_scores = lexical_results.get(query_id)
    if not doc_scores or not gold_ids:
        continue
    ranked_ids = [
        chunk_id
        for chunk_id, _ in sorted(
            doc_scores.items(), key=lambda item: item[1], reverse=True
        )
    ]
    if ranked_ids and ranked_ids[0] not in gold_ids:
        example_query_id = query_id
        break

if example_query_id is None:
    # Every lexical top-1 already matched a gold chunk; fall back to the
    # first qrels-bearing query so this cell still has something to show.
    example_query_id = next(iter(qrels_chunks))

gold_ids = {
    chunk_id
    for chunk_id, relevance in qrels_chunks[example_query_id].items()
    if relevance > 0
}

print("Query:")
print(odex_task.queries[example_query_id])
print(f"\nGold chunk id(s): {sorted(gold_ids)}")
for gold_id in sorted(gold_ids):
    gold_chunk = chunk_by_id[gold_id]
    gold_text = gold_chunk.text
    shown = gold_text[:DISPLAY_CHARS]
    suffix = "... [truncated for display]" if len(gold_text) > DISPLAY_CHARS else ""
    source_doc_id = path_to_doc_id.get(gold_chunk.path.name, "?")
    print(f"\n--- {gold_id} (from document {source_doc_id}, {len(gold_text)} chars) ---")
    print(shown + suffix)

print()
for channel_name, channel_results in (
    ("semantic", semantic_results),
    ("lexical", lexical_results),
    ("hybrid", hybrid_results),
):
    ranked = sorted(
        channel_results[example_query_id].items(),
        key=lambda item: item[1],
        reverse=True,
    )
    gold_rank = next(
        (
            rank
            for rank, (chunk_id, _) in enumerate(ranked, start=1)
            if chunk_id in gold_ids
        ),
        None,
    )
    print(f"--- {channel_name} (gold chunk ranked #{gold_rank}) ---")
    for rank, (chunk_id, score) in enumerate(ranked[:5], start=1):
        marker = " <-- gold" if chunk_id in gold_ids else ""
        print(f"  {rank}. {chunk_id}  score={score:.4f}{marker}")
    print()


Query:
capture final output of a chain of system commands `ps -ef | grep something | wc -l`

Gold chunk id(s): ['bc4eeaf531ed12d9']

--- bc4eeaf531ed12d9 (from document 19877, 1809 chars) ---
subprocess.check_output(args, *, stdin=None, stderr=None, shell=False, cwd=None, encoding=None, errors=None, universal_newlines=None, timeout=None, text=None, **other_popen_kwargs)  
Run command with arguments and return its output. If the return code was non-zero it raises a CalledProcessError. The CalledProcessError object will have the return code in the returncode attribute and any output in the output attribute. This is equivalent to: run(..., check=True, stdout=PIPE).stdout
 The arguments shown above are merely some common ones. The full function signature is largely the same as that of run() - most arguments are passed directly through to that interface. One API deviation from run() behavior exists: passing input=None will behave the same as input=b'' (or input='', depending on other argume

## 10. Save results


In [22]:
from datetime import datetime, timezone

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

run_record = {
    "benchmark": "codeRAG-bench (open-domain: odex)",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "embedding_model": MODEL_ID,
    "backend": selected_backend,
    "published_ndcg_at_10": PUBLISHED_NDCG_AT_10,
    "scores": odex_scores,
}

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_path = results_dir / f"{stamp}_odex.json"
output_path.write_text(json.dumps(run_record, indent=2))

print(f"Saved: {output_path.relative_to(root)}")


Saved: benchmarks/results/20260920T132711Z_odex.json


## 11. Release the model


In [23]:
embedding_session.close()

print("Embedding model released.")


Embedding model released.
